In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [4]:
train = pd.read_csv('/Users/albertzagibin/Documents/University/ML/I_Professional_AI/11/ИИ, продай мне этот аккумулятор!/X_train.csv')
y_train = pd.read_csv('/Users/albertzagibin/Documents/University/ML/I_Professional_AI/11/ИИ, продай мне этот аккумулятор!/y_train.csv')


In [3]:
train.head()

,Дата,Группа,№ Объявления,Формат,Размер изображения,Тип устройства,Пол,Категория таргетинга,Упоминание брендов,Уровень платежеспособности,Возраст,Заголовок,Текст
0,19.03.2024,Аккумуляторы Машина,M-9591367848,текстовый,без изображения,мобильные,мужской,Целевые запросы,Без упоминания вашего бренда,Остальные,45-54,Выберите авто аккумулятор с выгодой,Аккумуляторы для легковую машину в Иркутске. Д...
1,18.09.2023,Аккумуляторы Новый,M-9591367816,текстовый,без изображения,мобильные,мужской,не определено,не определено,6-10%,25-34,Новые Аккумуляторы,Centra Market - Большой выбор новых аккумулято...
2,07.02.2024,Аккумуляторы Иркутск,M-9591367718,текстовый,без изображения,мобильные,мужской,Широкие запросы,не определено,Остальные,старше 55,Большой выбор Авто аккумуляторов,Гарантия. Сервис. Свяжитесь со специалистом ил...
3,08.10.2023,Аккумуляторы Купить Цена,M-9591367860,графический,с изображением,десктоп,женский,Целевые запросы,не определено,6-10%,35-44,Купить аккумулятор с выгодой,Выгодные цены на аккумуляторы в Иркутске. Дост...
4,05.07.2023,Аккумуляторы Магазин,M-9591367826,текстовый,без изображения,мобильные,женский,Целевые запросы,не определено,Остальные,35-44,Большой магазин Авто аккумуляторов,Свяжитесь со специалистом или выберите аккумул...


In [ ]:
y_train.head()


,0
0,0.0
1,0.0
2,0.0
3,0.0
4,0.0


In [8]:
# --- Merge target y and basic inspection ---
# Create binary target column `y` from wCTR and merge with train features

print('Reading shapes:')
print('X_train:', train.shape)
print('y_train:', y_train.shape)

print('\nColumns X_train:')
print(train.columns.tolist())
print('\nColumns y_train:')
print(y_train.columns.tolist())

# Determine merge column automatically
merge_cols = list(set(train.columns) & set(y_train.columns))
if len(merge_cols) == 0:
    # fallback to index based merge
    print('\nNo shared column - merging on dataframe indices')
    train2 = train.reset_index(drop=False).rename(columns={'index': 'orig_index'})
    y2 = y_train.reset_index(drop=False).rename(columns={'index': 'orig_index'})
else:
    print('\nMerging on columns:', merge_cols)
    # pick the first shared column as merge key
    key = merge_cols[0]
    train2 = train.copy()
    y2 = y_train.copy()

# Create binary target y based on wCTR
if 'wCTR' in y2.columns:
    y2['y'] = (y2['wCTR'] > 0).astype(int)
else:
    # If wCTR missing, fallback to 'ctr' or other guesses
    alt = next((c for c in ['ctr', 'CTR', 'clicks', '0'] if c in y2.columns), None)
    if alt:
        y2['y'] = (y2[alt] > 0).astype(int)
    else:
        raise ValueError('Could not find wCTR/ctr column in y_train')

# Merge
if 'orig_index' in train2.columns and 'orig_index' in y2.columns:
    df = train2.merge(y2[['orig_index', 'y']], on='orig_index', how='left')
else:
    df = train2.merge(y2[[key, 'y']], on=key, how='left')

print('\nMerged shape:', df.shape)
print('\nTarget balance:')
print(df['y'].value_counts(dropna=False))

# Quick EDA on missing values and basic info
print('\nMissing values:')
print(df.isnull().mean().sort_values(ascending=False).head(20))

print('\nBasic info:')
print(df.info())

# Show top categorical columns
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
print('\nCategorical columns (sample):', cat_cols[:20])

# Show head
print('\nSample head:')
print(df.head(3))

Reading shapes:
X_train: (63445, 13)
y_train: (63445, 1)

Columns X_train:
['Дата', 'Группа', '№ Объявления', 'Формат', 'Размер изображения', 'Тип устройства', 'Пол', 'Категория таргетинга', 'Упоминание брендов', 'Уровень платежеспособности', 'Возраст', 'Заголовок', 'Текст']

Columns y_train:
['0']

No shared column - merging on dataframe indices

Merged shape: (63445, 15)

Target balance:
y
0    54135
1     9310
Name: count, dtype: int64

Missing values:
orig_index                    0.0
Дата                          0.0
Группа                        0.0
№ Объявления                  0.0
Формат                        0.0
Размер изображения            0.0
Тип устройства                0.0
Пол                           0.0
Категория таргетинга          0.0
Упоминание брендов            0.0
Уровень платежеспособности    0.0
Возраст                       0.0
Заголовок                     0.0
Текст                         0.0
y                             0.0
dtype: float64

Basic info:
<c

In [13]:
df.head()

,orig_index,Дата,Группа,№ Объявления,Формат,Размер изображения,Тип устройства,Пол,Категория таргетинга,Упоминание брендов,Уровень платежеспособности,Возраст,Заголовок,Текст,y
0,0,19.03.2024,Аккумуляторы Машина,M-9591367848,текстовый,без изображения,мобильные,мужской,Целевые запросы,Без упоминания вашего бренда,Остальные,45-54,Выберите авто аккумулятор с выгодой,Аккумуляторы для легковую машину в Иркутске. Д...,0
1,1,18.09.2023,Аккумуляторы Новый,M-9591367816,текстовый,без изображения,мобильные,мужской,не определено,не определено,6-10%,25-34,Новые Аккумуляторы,Centra Market - Большой выбор новых аккумулято...,0
2,2,07.02.2024,Аккумуляторы Иркутск,M-9591367718,текстовый,без изображения,мобильные,мужской,Широкие запросы,не определено,Остальные,старше 55,Большой выбор Авто аккумуляторов,Гарантия. Сервис. Свяжитесь со специалистом ил...,0
3,3,08.10.2023,Аккумуляторы Купить Цена,M-9591367860,графический,с изображением,десктоп,женский,Целевые запросы,не определено,6-10%,35-44,Купить аккумулятор с выгодой,Выгодные цены на аккумуляторы в Иркутске. Дост...,0
4,4,05.07.2023,Аккумуляторы Магазин,M-9591367826,текстовый,без изображения,мобильные,женский,Целевые запросы,не определено,Остальные,35-44,Большой магазин Авто аккумуляторов,Свяжитесь со специалистом или выберите аккумул...,0


In [ ]:
# --- Feature engineering (simple) ---
# Create features: datetime parts, text length features, freq-enc cats, brand flags

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Make a working copy
D = df.copy()

# Datetime features: attempt to find a date column
date_cols = [c for c in D.columns if 'date' in c.lower() or 'time' in c.lower()]
if len(date_cols) > 0:
    date_col = date_cols[0]
    print('Using date column for features:', date_col)
    D[date_col] = pd.to_datetime(D[date_col], errors='coerce')
    D['date_day'] = D[date_col].dt.day.fillna(-1).astype(int)
    D['date_month'] = D[date_col].dt.month.fillna(-1).astype(int)
    D['date_weekday'] = D[date_col].dt.weekday.fillna(-1).astype(int)
    D['date_hour'] = D[date_col].dt.hour.fillna(-1).astype(int)
else:
    print('No date column found')

# Text features
text_cols = [c for c in D.columns if any(x in c.lower() for x in ['text', 'title', 'description'])]
print('Text columns detected:', text_cols)
for c in text_cols:
    D[c + '_len'] = D[c].fillna('').astype(str).apply(len)
    D[c + '_n_words'] = D[c].fillna('').astype(str).apply(lambda x: len(x.split()))

# Brand presence flag
brand_cols = [c for c in D.columns if 'brand' in c.lower()]
if len(brand_cols) > 0:
    b = brand_cols[0]
    D['has_brand'] = D[b].notnull().astype(int)
    print('Brand column found:', b)
else:
    print('No brand column found')

# Device/format frequency encoding
cat_candidates = [c for c in D.columns if D[c].dtype == 'object']
cat_candidates = [c for c in cat_candidates if c not in text_cols]
print('Categorical columns candidate:', cat_candidates[:20])

# Frequency encoding helper
for c in cat_candidates:
    freq = D[c].fillna('missing').value_counts(normalize=True)
    D[c + '_freq'] = D[c].fillna('missing').map(freq).astype(float)

# Build final feature list
feat_cols = []
# numeric columns
num_cols = D.select_dtypes(include=[np.number]).columns.tolist()
# Remove target and any id columns
num_cols = [c for c in num_cols if c not in ['y'] and not c.endswith('_freq') and not c in ['orig_index']]
# Add calculated features explicitly
calc_feats = [c for c in D.columns if c.endswith('_len') or c.endswith('_n_words') or c.startswith('date_') or c == 'has_brand']
feat_cols = calc_feats + [c for c in D.columns if c.endswith('_freq')]

# Deduplicate
feat_cols = [c for c in feat_cols if c in D.columns]

print('\nNum numeric feature sample:', num_cols[:10])
print('Computed features sample:', feat_cols[:30])

# Fill missing
D[feat_cols] = D[feat_cols].fillna(-1)

# Split
train_df, val_df = train_test_split(D, stratify=D['y'], test_size=0.2, random_state=42)
X_train = train_df[feat_cols]
X_val = val_df[feat_cols]
y_train_bin = train_df['y']
y_val_bin = val_df['y']

print('\nTrain/Val shapes:', X_train.shape, X_val.shape)
print('\nPositive rate (train/val):', y_train_bin.mean(), y_val_bin.mean())

No date column found
Text columns detected: []
No brand column found
Categorical columns candidate: ['Дата', 'Группа', '№ Объявления', 'Формат', 'Размер изображения', 'Тип устройства', 'Пол', 'Категория таргетинга', 'Упоминание брендов', 'Уровень платежеспособности', 'Возраст', 'Заголовок', 'Текст']

Num numeric feature sample: []
Computed features sample: ['Дата_freq', 'Группа_freq', '№ Объявления_freq', 'Формат_freq', 'Размер изображения_freq', 'Тип устройства_freq', 'Пол_freq', 'Категория таргетинга_freq', 'Упоминание брендов_freq', 'Уровень платежеспособности_freq', 'Возраст_freq', 'Заголовок_freq', 'Текст_freq']

Train/Val shapes: (50756, 13) (12689, 13)

Positive rate (train/val): 0.14674127196784617 0.14674127196784617


In [10]:
# --- Baseline modeling: LogisticRegression + XGB/RandomForest ---
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score, confusion_matrix, classification_report


def evaluate_model(model, X_val, y_val, name='model'):
    y_pred = model.predict(X_val)
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_val)[:, 1]
    else:
        y_proba = model.decision_function(X_val)
    print('\n=== Evaluation for', name, '===')
    print('ROC AUC:', roc_auc_score(y_val, y_proba))
    print('Accuracy:', accuracy_score(y_val, y_pred))
    print('F1:', f1_score(y_val, y_pred))
    print('Precision:', precision_score(y_val, y_pred))
    print('Recall:', recall_score(y_val, y_pred))
    print('\nClassification Report:\n', classification_report(y_val, y_pred))
    print('Confusion Matrix:\n', confusion_matrix(y_val, y_pred))


# Pipeline for logistic regression
log_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=-1)),
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', solver='liblinear'))
])

log_pipe.fit(X_train, y_train_bin)

evaluate_model(log_pipe, X_val, y_val_bin, 'LogisticRegression')

# Tree based model: try xgboost first
try:
    import xgboost as xgb
    xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42, n_jobs=4)
    xgb_model.fit(X_train, y_train_bin)
    evaluate_model(xgb_model, X_val, y_val_bin, 'XGBoost')
    tree_model = xgb_model
except Exception as e:
    print('XGBoost not available or failed: ', e)
    print('Falling back to RandomForest')
    rf_model = RandomForestClassifier(n_estimators=200, n_jobs=4, random_state=42, class_weight='balanced')
    rf_model.fit(X_train, y_train_bin)
    evaluate_model(rf_model, X_val, y_val_bin, 'RandomForest')
    tree_model = rf_model

# Feature importances if tree model
try:
    importances = tree_model.feature_importances_
    feat_imp = pd.Series(importances, index=feat_cols).sort_values(ascending=False)
    print('\nTop features (tree-based importance):')
    print(feat_imp.head(30))
except Exception as e:
    print('Could not compute feature importances:', e)

# Save predictions and sample submission
val_preds = pd.DataFrame({'y_true': y_val_bin.values, 'y_pred': tree_model.predict(X_val), 'y_proba': tree_model.predict_proba(X_val)[:, 1]})
print('\nSample predictions on validation:')
print(val_preds.head())

# Save predictions to disk
val_preds.to_csv('baseline_val_preds.csv', index=False)
print('Saved baseline_val_preds.csv')

/Users/albertzagibin/Documents/University/ML/venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/albertzagibin/Documents/University/ML/venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/albertzagibin/Documents/University/ML/venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/albertzagibin/Documents/University/ML/venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/albertzagibin/Documents/University/ML/venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/albertzagibin/Documents/University/ML/venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid 


=== Evaluation for LogisticRegression ===
ROC AUC: 0.6285806399385234
Accuracy: 0.5674994089368744
F1: 0.3010697911360163
Precision: 0.1973288814691152
Recall: 0.6348012889366272

Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.56      0.69     10827
           1       0.20      0.63      0.30      1862

    accuracy                           0.57     12689
   macro avg       0.55      0.60      0.49     12689
weighted avg       0.80      0.57      0.63     12689

Confusion Matrix:
 [[6019 4808]
 [ 680 1182]]

=== Evaluation for XGBoost ===
ROC AUC: 0.6741696649492948
Accuracy: 0.8526282606982426
F1: 0.050761421319796954
Precision: 0.46296296296296297
Recall: 0.02685284640171858

Classification Report:
               precision    recall  f1-score   support

           0       0.86      0.99      0.92     10827
           1       0.46      0.03      0.05      1862

    accuracy                           0.85     12689
   mac

In [11]:
# Save models and results
import joblib

# Save tree model if exists
try:
    joblib.dump(tree_model, 'baseline_tree_model.pkl')
    print('Saved baseline_tree_model.pkl')
except Exception as e:
    print('Could not save tree model:', e)

# Save logistic regression
try:
    joblib.dump(log_pipe, 'baseline_logistic_model.pkl')
    print('Saved baseline_logistic_model.pkl')
except Exception as e:
    print('Could not save logistic model:', e)

# Print the class balance and a small result summary
print('\nFinal positive rate in dataset:', df['y'].mean())
print('\nValidation sample predictions saved to baseline_val_preds.csv')

Saved baseline_tree_model.pkl
Saved baseline_logistic_model.pkl

Final positive rate in dataset: 0.14674127196784617

Validation sample predictions saved to baseline_val_preds.csv
